# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIRˆ² colorectal cancer survivors dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, which enables programmatic access to Croissant-standardized datasets.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`. The Croissant URL points to a JSON-LD description of the dataset, including its schema and record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Instantiate the Dataset object
dataset = mlc.Dataset(croissant_url)

# Get and print friendly dataset summary
metadata = dataset.metadata
print("\nDataset Name:", metadata.name)
print("\nDescription:", metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This step helps us understand what tables (record sets) and which variables/columns (fields) the dataset contains.

Each entity is referenced by its `@id`, following the Croissant and FAIR2 standards.

In [ ]:
# List details for all record sets in the dataset
print("Available record sets and their fields/columns:\n")

record_set_infos = []
for record_set in metadata.record_sets:
    print(f"Record set: '{record_set.name}' (ID: {record_set.id})")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - {getattr(field, 'name', '<no name>')} (ID: {field.id}, Type: {getattr(field, 'data_type', '')})")
            record_set_infos.append((record_set.id, field.id, getattr(field, 'name', None)))
    else:
        print("  (No fields declared)")
    print()

## 3. Data Extraction
Load data from each available record set into pandas DataFrames. Use the record set and field `@id`s previously listed.

*Note: All extractions and keys reference Croissant-standard `@id`s for absolute clarity, transparency, and reproducibility.*

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
print(f"Record set @ids: {record_set_ids}\n")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set '{record_set_id}' with columns: {list(df.columns)[:8]}{'...' if len(df.columns) > 8 else ''}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

# Preview the first (main) record set's DataFrame
primary_record_set_id = record_set_ids[0] if record_set_ids else None
if primary_record_set_id in dataframes:
    print(f"\nColumns in main record set ({primary_record_set_id}):")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
* Filtering records with specific criteria
* Normalizing numeric fields
* Grouping/categorizing by key attributes

All filtering/grouping operations again use the appropriate field `@id` references.

In [ ]:
# Choose the main record set and look for suitable numeric and categorical fields
if not primary_record_set_id or primary_record_set_id not in dataframes:
    print("No data available for EDA!")
else:
    df = dataframes[primary_record_set_id]

    # Print non-null columns and guess a numeric/categorical one for demonstration
    print("Columns and sample values (first row):")
    for col in df.columns:
        print(f"{col} - sample: {df[col].iloc[0] if not df.empty else None}")

    # Let's try to pick a numeric field; fallback to arbitrary if structure is unknown.
    # We'll search for int/float-looking columns based on dtype or column name
    possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not possible_numeric_fields:
        # fallback if all object: try common names
        possible_numeric_fields = [col for col in df.columns if any(word in col.lower() for word in ['age', 'interval', 'years', 'number', 'count'])]
    numeric_field_id = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]

    print(f"\nUsing numeric field for analysis: {numeric_field_id}")
    # Apply a simple threshold filter (show only if field is truly numeric for demo)
    try:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id])
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (median):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    except Exception as e:
        print(f"Numeric analysis skipped, could not convert ({numeric_field_id}): {e}")

    # Group by a likely categorical field
    possible_group_fields = [col for col in df.columns if any(k in col.lower() for k in ['sex', 'msi', 'status', 'site', 'dist', 'category', 'group'])]
    group_field_id = possible_group_fields[0] if possible_group_fields else None
    if group_field_id:
        print(f"\nGrouping by field: {group_field_id}")
        try:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Group averages of {numeric_field_id} by {group_field_id}:")
            print(grouped)
        except Exception as e:
            print(f"Could not group by {group_field_id}: {e}")
    else:
        print("No categorical field found for grouping.")

## 5. Visualization
Visualize one or more distributions or relationships between fields in the dataset.

*Example: Plotting the numeric field's distribution, and a boxplot by group.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_record_set_id in dataframes and not df.empty:
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrates how to work with a Croissant-standardized clinical dataset using `mlcroissant`:

- **Data discovery:** All entities and variables are referenced by their stable `@id` values, ensuring clarity and reproducibility.
- **Flexible exploration:** Data can be loaded, filtered, normalized, and grouped using standard Python and pandas workflows.
- **Visualization:** The distribution and categorical relationships of key clinical variables can be visualized to support downstream analyses.

For more advanced analytics and domain-specific insights, consult the data dictionary accompanying the FAIRˆ² dataset, and use explicit `@id` fields to reference data elements in your pipelines.